# Creating a Random Forest ML Model

This notebook goes through how to rapidly protype a Random Forest model with ```ml_fw```. It follows the methodology of Murphy et al. 2024 and reproduces the FISM2/Geo model. 

It doesn't do a grid search for the Random Forest parameters as it's very time consuming. It does however go through the generation of the correlation matrices (Figure 2 b and c), plotting of the case-studies (Figure 5), and the residual analysis (Figure 6)



In [28]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

path_dat = f'D:\\data\\SatDensities\\satdrag_database_grace_B.hdf5'
path_oos = f'D:\\data\\SatDensities\\satdrag_database_grace_CHAMP.hdf5'
path_mod = os.path.normpath(os.getcwd()+'/../ml_fw/') # assumes current working directory is the ml_fw/Notebooks directory

# add the ml_fw module to Python Path and import what we need
sys.path.append(os.path.dirname(path_mod))

In [29]:
from ml_fw import profile as pro
from ml_fw import data_io as dio
from ml_fw import ml_mod as ml

In [30]:
# define the columns that we will be using in the model
# the time columns are helpful when investigating case studies
# or model performance for categorical variables (storm/storm phase)
df_cols = {
    'feat_col': ['1300_02','43000_09','85550_13','94400_18',
                  'SYM_H index','AE','SatLat'],
    'y_col': ['400kmDensity'],
    'log_col': ['1300_02', '43000_09', '85550_13', '94400_18'],
    'lt_col': ['SatMagLT'],
    't_col': ['DateTime', 'storm', 'storm phase']
    }

# define RF model parameters
rnd=17
rf_params = {
    "n_estimators": 500,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf":5,
    "warm_start":False,
    "oob_score":True,
    "random_state": rnd,
    "max_features":0.5,
    "n_jobs":10
    }

In [ ]:
# read in the data
# and drop na values
dat = pd.read_hdf(path_dat)

dat = dat[df_cols['feat_col']+df_cols['y_col']+df_cols['lt_col']+df_cols['t_col']].dropna()

# use create from data_io to generate two 
# dataframes:
# feature and time DataFrame
# target/y DataFrame
ml_f, ml_y = dio.create(dat,**df_cols)

# convert density to a more appropriate
# range to avoid errors (e.g., target too 
# small)
ml_y = ml_y*10**12 

# delete that data for space
del dat

c:\Users\krmurph1\Anaconda3\envs\satdrag\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\krmurph1\Anaconda3\envs\satdrag\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [32]:
# create train test splits
train_x, test_x, train_y, test_y = train_test_split(ml_f, ml_y, 
                                                        test_size=0.3, 
                                                        random_state=rnd)

# pull and drop the time columns from the x data
train_t = train_x[df_cols['t_col']]
test_t = test_x[df_cols['t_col']]

train_x = train_x.drop(columns=df_cols['t_col'])
test_x = test_x.drop(columns=df_cols['t_col'])

In [33]:
# train the random forest model
rfr = RandomForestRegressor(**rf_params)
rfr.fit(train_x, train_y.values.ravel())

RandomForestRegressor(max_features=0.5, min_samples_leaf=5, n_estimators=500,
                      n_jobs=10, oob_score=True, random_state=17)

In [34]:
# predict the y-values from the train
# data and combine into a single
# DataFrame
pred_tr = rfr.predict(train_x)

In [35]:
train_d = train_x.join([train_y,train_t], how='left')
train_d[df_cols['y_col'][0]+'_pred'] = pred_tr


In [36]:
# determine predictions for
# out-of-sample dataset
oos_dat = pd.read_hdf(path_oos)

oos_f, oos_y = dio.create(oos_dat,**df_cols)
oos_y = oos_y*10**12

oos_d = oos_f.join([oos_y])

oos_d[df_cols['y_col'][0]+'_pred'] = rfr.predict(oos_f.drop(columns=df_cols['t_col']))

# delete the full DataFrame for space
del oos_dat



c:\Users\krmurph1\Anaconda3\envs\satdrag\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\krmurph1\Anaconda3\envs\satdrag\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [59]:
(new[-2]['400kmDensity'] - oos_d['400kmDensity']).describe()

count    790123.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: 400kmDensity, dtype: float64

In [45]:
new[-2].shape

(790123, 14)

In [46]:
oos_d.shape

(790123, 14)